Notation is for device A, B $P_{a, b}$ where subscripts denote number of photons detected on device A and B respectively. So $P_{1, 0}$ is device a triggering and b not. No conditional probabilties here  

$O_{a}$ is the occupancy of device A, meaning the (independent) probability device A triggers (1PE) when laser fires. Similarly for device B.

$P_{xab}$ is probability A triggers B (because of ExCT)

$$P_{0,0} = (1-O_a)(1-O_b)$$
$$P_{1,0} = O_a(1-O_b)(1-P_{xab})$$
$$P_{0,1} = (1-O_a)O_b(1-P_{xba})$$
$$P_{1,1} = O_aO_b + O_a(1-O_b)P_{xab} + (1-O_a)O_bP_{xba}$$

given measurements, we can solve for $P_{xab}$ and $P_{xba}$

$$P_{xab} = 1-\frac{P_{1,0}}{O_a(1-O_b)} $$

In [24]:
import numpy as np

def run_trigger_mc(N, Poa, Pob, Pxab, Pxba, seed=None):
    """
    Simulate N independent trials of two devices A and B.
    
    Process per trial:
    1. Device A triggers independently with probability Poa
    2. Device B triggers independently with probability Pob
    3. If A triggered (step 1), it may also trigger B with probability Pxab (crosstalk A→B)
    4. If B triggered (step 1), it may also trigger A with probability Pxba (crosstalk B→A)
    
    Returns:
        A_final: bool array (N,) — whether A ended up triggered
        B_final: bool array (N,) — whether B ended up triggered
    """
    rng = np.random.default_rng(seed)
    
    # Step 1: independent triggers
    A_self = rng.random(N) < Poa  # A triggers on its own
    B_self = rng.random(N) < Pob  # B triggers on its own
    
    # Step 2: crosstalk (only matters if the source device triggered)
    # A→B crosstalk: if A triggered, may also trigger B
    A_triggers_B = A_self & (rng.random(N) < Pxab)
    # B→A crosstalk: if B triggered, may also trigger A
    B_triggers_A = B_self & (rng.random(N) < Pxba)
    
    # Final state: triggered if self-triggered OR triggered by crosstalk
    A_final = A_self | B_triggers_A
    B_final = B_self | A_triggers_B
    
    return A_final, B_final


def Calc_Pxab(P_10, Oa, Ob):
    """
    Given observed probabilities:
    P_10: Probability that only A triggered
    Oa: Probability that A triggered (regardless of B)
    Ob: Probability that B triggered (regardless of A)
    
    Calculate the crosstalk probability Pxab (A triggering B).
    
    Returns:
        Pxab: Crosstalk probability from A to B
    """
    return 1 - (P_10 / (Oa * (1 - Ob)))

In [39]:
# Parameters
N = 10_000_00  # 10 million trials
Poa = 0.10     # 5% chance A triggers on its own
Pob = 0.50     # 5% chance B triggers on its own
Pxab = 0.01     # 10% chance A triggers B (given A fired)
Pxba = 0.01     # 10% chance B triggers A (given B fired)

A, B = run_trigger_mc(N, Poa, Pob, Pxab, Pxba, seed=None)

# print(A)
# print(B)

# Count outcome categories
neither = np.sum(~A & ~B)/N
only_A  = np.sum(A & ~B)/N # this is our observed P_10, given we normalize by N_laser
only_B  = np.sum(~A & B)/N
both    = np.sum(A & B)/N

print(f"N = {N:,}")
print(f"Neither: {neither:,} ({100*neither:.4f}%)")
print(f"Only A:  {only_A:,} ({100*only_A:.4f}%)")
print(f"Only B:  {only_B:,} ({100*only_B:.4f}%)")
print(f"Both:    {both:,} ({100*both:.4f}%)")

# Analytical expectation (approximate, ignoring double-crosstalk edge cases):
# P(A) ≈ Poa + Pob*Pxba - Poa*Pob*Pxba
# P(B) ≈ Pob + Poa*Pxab - Poa*Pob*Pxab
# P(both) ≈ Poa*Pob + Poa*Pxab + Pob*Pxba - ...
print("\n--- Quick sanity check ---")
print(f"Fraction A triggered: {A.mean():.6f}")
print(f"Fraction B triggered: {B.mean():.6f}")
print(f"Fraction both triggered: {(A & B).mean():.6f}")

print("\n--- Calculating Pxab from observed probabilities ---")
P_xab = Calc_Pxab(only_A, Poa, Pob)
print(f"Calculated Pxab: {P_xab:.6f} (input value was {Pxab})")

N = 1,000,000
Neither: 0.450178 (45.0178%)
Only A:  0.04933 (4.9330%)
Only B:  0.445029 (44.5029%)
Both:    0.055463 (5.5463%)

--- Quick sanity check ---
Fraction A triggered: 0.104793
Fraction B triggered: 0.500492
Fraction both triggered: 0.055463

--- Calculating Pxab from observed probabilities ---
Calculated Pxab: 0.013400 (input value was 0.01)


---
Let us now do this conditionally. Probability B has NO trigger, given A triggered.

If we actually follow the strict definition, we get a different formula than that reported by Lei:  

$$P(B=0\mid A=1) = \frac{P_{1,0}}{P_{1,0} + P_{1,1}}.$$

Using the NON-CONDITIONAL formulas above,
$$ P_{1,0} = O_a(1-O_b)(1-P_{xab}) $$
$$ P_{1,1} = O_aO_b + O_a(1-O_b)P_{xab} + (1-O_a)O_bP_{xba}. $$

The denominator simplifies to
$$
P_{1,0} + P_{1,1} = O_a + (1-O_a)O_bP_{xba}.
$$

So the conditional probability is
$$
P(B=0\mid A=1)
= \frac{O_a(1-O_b)(1-P_{xab})}{O_a + (1-O_a)O_bP_{xba}}$$

this BRINGS IN the dependence on P_xba, which we want to avoid. This is not surprising, BECAUSE ITS CONDITIONAL COUNTING WHICH IS THE EXACT SAME ISSUE WE HAD ON THE LOLX1 PAPER, WHICH BROUGHT IN BOTH FORWARD AND BACKWARDS PROCESSES. So don't tell me 'I dont understand the process'